# BI10 — Challenge 1 | Bước 3.1 & 3.2
**3.1** Đối chiếu `CONTRACT.csv` gốc với `CONTRACT_clean.csv` theo mã case; không tự hợp thức hóa thu nhập xung đột.  
**3.2** Trích xuất 6 đặc trưng ứng viên ở cấp khách hàng từ dữ liệu clean; chưa xây score hoặc gán trọng số.

| Nhóm | Feature | Nội dung |
|---|---|---|
| Capacity | A1 | ODSR ứng viên nếu kỳ thu nhập là tháng |
| Capacity | A2 | Thu nhập còn lại sau trả góp quan sát được (ứng viên) |
| Capacity | A3 | Số hợp đồng có trạng thái Active |
| Payment | P1 | Tỷ lệ case có DUE trong nhóm trạng thái đã giải quyết (proxy) |
| Payment | P2 | Số cờ DUE trung bình trong các case có DUE |
| Payment | P3 | Lịch sử ghi nhận Written off |

> `ANNUITY` được mô tả theo tháng nhưng kỳ `AMT_INCOME_MAIN`, thời điểm trạng thái và mức trưởng thành các cờ DUE chưa được xác minh. Đầu ra chứa **candidate features + quality flags**, không phải DTI/PD được công nhận.

### Cell 1 — Đường dẫn, tài nguyên và kết nối độc lập

In [1]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import duckdb
import pandas as pd
from IPython.display import display

S31_CUSTOM_CLEAN_DIR = None  # Điền đường dẫn clean_data nếu notebook đặt ở nơi khác.
S31_INCOME_IS_MONTHLY_CONFIRMED = False  # Chỉ đổi thành True nếu có xác nhận từ data owner.
S31_ACCEPT_ZERO_PLUS_POSITIVE = True     # Cho phép thu nhập dương đồng nhất + bản ghi 0, kèm quality flag.

roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
choices = ([Path(S31_CUSTOM_CLEAN_DIR)] if S31_CUSTOM_CLEAN_DIR else
           [r / sub for r in roots for sub in ('clean_data', 'BI10_ROUND02_DATASET/clean_data')])
S31_CLEAN = next((p.resolve() for p in choices
                  if (p / 'CONTRACT_clean.csv').is_file() and (p / 'CUSTOMER_clean.csv').is_file()), None)
if S31_CLEAN is None:
    raise FileNotFoundError('Không tìm thấy CONTRACT_clean.csv và CUSTOMER_clean.csv; khai báo S31_CUSTOM_CLEAN_DIR.')
S31_RAW = S31_CLEAN.parent / 'CONTRACT.csv'
if not S31_RAW.is_file():
    raise FileNotFoundError(f'Không tìm thấy CONTRACT.csv gốc để đối chiếu: {S31_RAW}')

S31_OUT = S31_CLEAN.parent / 'C1_STEP3_OUTPUT' / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:6])
S31_TABLES, S31_TMP = S31_OUT / 'tables', S31_OUT / 'scratch'
for p in (S31_TABLES, S31_TMP): p.mkdir(parents=True, exist_ok=True)

# Không mở bi10_eda.duckdb hoặc audit_cache.duckdb; tránh xung đột với notebook khác.
S31_DB = duckdb.connect(':memory:')
S31_DB.execute('SET threads=4')
S31_DB.execute("SET memory_limit='4GB'")
S31_DB.execute('SET preserve_insertion_order=false')
S31_DB.execute("SET temp_directory='" + S31_TMP.as_posix().replace("'", "''") + "'")

def s31_q(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"

def s31_csv_view(name, path):
    S31_DB.execute(f'''CREATE OR REPLACE TEMP VIEW {name} AS SELECT * FROM read_csv(
        {s31_q(path)}, header=true, all_varchar=true,
        delim=',', quote='"', escape='"', ignore_errors=false)''')

def s31_export_csv(filename, sql):
    target = S31_TABLES / filename
    S31_DB.execute(f"COPY ({sql}) TO {s31_q(target)} (HEADER, DELIMITER ',')")
    return target

print('CLEAN:', S31_CLEAN)
print('RAW:', S31_RAW)
print('OUTPUT:', S31_OUT)

CLEAN: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\clean_data
RAW: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\CONTRACT.csv
OUTPUT: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\C1_STEP3_OUTPUT\run_20261018_091425_849a72


### Cell 2 — Kiểm tra cấu trúc và tính duy nhất sau cleaning

In [2]:
s31_csv_view('s31_raw_contract', S31_RAW)
s31_csv_view('s31_clean_contract', S31_CLEAN / 'CONTRACT_clean.csv')
s31_csv_view('s31_clean_customer', S31_CLEAN / 'CUSTOMER_clean.csv')

required_contract = {'SKP_CREDIT_CASE','SKP_CLIENT','PRODUCT','NAME_CREDIT_STATUS',
                     'AMT_INCOME_MAIN','ANNUITY','APPLY_CONTRACT_TIME',
                     'FIRST_DUE','SECOND_DUE','THIRD_DUE','FOURTH_DUE'}
for table in ('s31_raw_contract','s31_clean_contract'):
    columns = {c[0].upper() for c in S31_DB.execute(f'SELECT * FROM {table} LIMIT 0').description}
    if missing := required_contract - columns:
        raise ValueError(f'{table} thiếu cột: {sorted(missing)}')
if 'SKP_CLIENT' not in {c[0].upper() for c in S31_DB.execute('SELECT * FROM s31_clean_customer LIMIT 0').description}:
    raise ValueError('CUSTOMER_clean.csv thiếu SKP_CLIENT')

check = S31_DB.execute('''
WITH a AS (SELECT COUNT(*) AS n, COUNT(DISTINCT NULLIF(TRIM(SKP_CREDIT_CASE),'')) AS k,
                  COUNT(*) FILTER (WHERE NULLIF(TRIM(SKP_CREDIT_CASE),'') IS NULL) AS missing_id
           FROM s31_clean_contract),
b AS (SELECT COUNT(*) AS n, COUNT(DISTINCT NULLIF(TRIM(SKP_CLIENT),'')) AS k,
                  COUNT(*) FILTER (WHERE NULLIF(TRIM(SKP_CLIENT),'') IS NULL) AS missing_id
           FROM s31_clean_customer)
SELECT a.n AS n_clean_cases, a.k AS unique_case_ids, a.missing_id AS missing_case_ids,
       b.n AS n_clean_customers, b.k AS unique_client_ids, b.missing_id AS missing_client_ids
FROM a CROSS JOIN b
''').df()
display(check)
if (check.n_clean_cases.iloc[0] != check.unique_case_ids.iloc[0]
    or check.missing_case_ids.iloc[0] > 0
    or check.n_clean_customers.iloc[0] != check.unique_client_ids.iloc[0]
    or check.missing_client_ids.iloc[0] > 0):
    raise ValueError('File clean không phải 1 case / 1 dòng, hoặc Customer không phải 1 khách / 1 dòng. Dừng để tránh sai lệch.')

,n_clean_cases,unique_case_ids,missing_case_ids,n_clean_customers,unique_client_ids,missing_client_ids
0,4280377,4280377,0,1000528,1000528,0


### Cell 3 — Bước 3.1: Đối chiếu giá trị thu nhập của từng case trước–sau cleaning

In [3]:
S31_RAW_PROFILE = S31_TMP / 'raw_income_profile.parquet'
S31_DB.execute(f'''
COPY (
WITH raw AS (
    SELECT NULLIF(TRIM(SKP_CREDIT_CASE),'') AS case_id,
           NULLIF(TRIM(SKP_CLIENT),'') AS client_id,
           NULLIF(UPPER(TRIM(NAME_CREDIT_STATUS)),'') AS status_name,
           TRY_CAST(NULLIF(TRIM(AMT_INCOME_MAIN),'') AS DOUBLE) AS income
    FROM s31_raw_contract
)
SELECT case_id, COUNT(*) AS raw_records,
       MIN(client_id) AS min_client, MAX(client_id) AS max_client,
       COUNT(*) FILTER (WHERE client_id IS NULL) AS missing_raw_clients,
       MIN(status_name) AS min_status, MAX(status_name) AS max_status,
       COUNT(*) FILTER (WHERE income > 0) AS n_positive,
       COUNT(*) FILTER (WHERE income = 0) AS n_zero,
       COUNT(*) FILTER (WHERE income IS NULL OR income < 0) AS n_invalid_or_missing,
       MIN(income) FILTER (WHERE income > 0) AS min_positive,
       MAX(income) FILTER (WHERE income > 0) AS max_positive
FROM raw WHERE case_id IS NOT NULL GROUP BY case_id
) TO {s31_q(S31_RAW_PROFILE)} (FORMAT PARQUET, COMPRESSION ZSTD)
''')
S31_DB.execute(f'CREATE OR REPLACE TEMP VIEW s31_raw_profile AS SELECT * FROM read_parquet({s31_q(S31_RAW_PROFILE)})')

S31_CASE = S31_TMP / 'reconciled_cases.parquet'
S31_DB.execute(f'''
COPY (
WITH clean AS (
    SELECT TRIM(SKP_CREDIT_CASE) AS case_id, TRIM(SKP_CLIENT) AS client_id,
           NULLIF(UPPER(TRIM(PRODUCT)),'') AS product,
           NULLIF(UPPER(TRIM(NAME_CREDIT_STATUS)),'') AS credit_status,
           TRY_CAST(NULLIF(TRIM(AMT_INCOME_MAIN),'') AS DOUBLE) AS reported_income,
           TRY_CAST(NULLIF(TRIM(ANNUITY),'') AS DOUBLE) AS annuity,
           TRY_CAST(NULLIF(TRIM(APPLY_CONTRACT_TIME),'') AS TIMESTAMP) AS application_time,
           TRY_CAST(NULLIF(TRIM(FIRST_DUE),'') AS INTEGER) AS d1,
           TRY_CAST(NULLIF(TRIM(SECOND_DUE),'') AS INTEGER) AS d2,
           TRY_CAST(NULLIF(TRIM(THIRD_DUE),'') AS INTEGER) AS d3,
           TRY_CAST(NULLIF(TRIM(FOURTH_DUE),'') AS INTEGER) AS d4
    FROM s31_clean_contract
), joined AS (
    SELECT c.*, r.raw_records, r.n_positive, r.n_zero, r.n_invalid_or_missing,
           r.min_positive, r.max_positive,
           COALESCE(r.min_client=c.client_id AND r.max_client=c.client_id
                    AND r.missing_raw_clients=0, FALSE) AS client_link_ok,
           COALESCE(r.min_status=c.credit_status AND r.max_status=c.credit_status, FALSE) AS status_match,
           CASE
             WHEN r.case_id IS NULL THEN 'CASE_NOT_IN_RAW'
             WHEN NOT COALESCE(r.min_client=c.client_id AND r.max_client=c.client_id
                               AND r.missing_raw_clients=0,FALSE) THEN 'CLIENT_MISMATCH'
             WHEN r.n_positive=0 THEN 'NO_RAW_POSITIVE_INCOME'
             WHEN r.min_positive<>r.max_positive THEN 'MULTIPLE_RAW_POSITIVE_VALUES'
             WHEN c.reported_income IS NULL OR c.reported_income<=0 THEN 'NO_CLEAN_POSITIVE_INCOME'
             WHEN ABS(c.reported_income-r.min_positive)>GREATEST(0.01,ABS(r.min_positive)*1e-7)
                  THEN 'CLEAN_RAW_INCOME_MISMATCH'
             WHEN r.n_zero>0 OR r.n_invalid_or_missing>0 THEN 'ZERO_OR_MISSING_PLUS_POSITIVE'
             ELSE 'CONSISTENT_POSITIVE'
           END AS income_recon_class
    FROM clean c LEFT JOIN s31_raw_profile r ON c.case_id=r.case_id
), flags AS (
    SELECT *,
      (client_link_ok AND status_match) AS case_integrity_ok,
      (income_recon_class='CONSISTENT_POSITIVE'
        OR (income_recon_class='ZERO_OR_MISSING_PLUS_POSITIVE'
            AND {str(S31_ACCEPT_ZERO_PLUS_POSITIVE).upper()})) AS income_candidate_usable,
      (credit_status IN ('ACTIVE','FINISHED','PAID OFF','SIGNED','WRITTEN OFF')) AS signed_status,
      (credit_status IN ('FINISHED','PAID OFF','WRITTEN OFF')) AS resolved_status_proxy,
      (d1 IN (0,1) AND d2 IN (0,1) AND d3 IN (0,1) AND d4 IN (0,1)) AS due_flags_valid,
      CASE WHEN d1 IN (0,1) AND d2 IN (0,1) AND d3 IN (0,1) AND d4 IN (0,1)
           THEN d1+d2+d3+d4 END AS due_flag_count
    FROM joined
)
SELECT *, CASE WHEN due_flags_valid THEN CAST(due_flag_count>0 AS INTEGER) END AS any_due
FROM flags
) TO {s31_q(S31_CASE)} (FORMAT PARQUET, COMPRESSION ZSTD)
''')
S31_DB.execute(f'CREATE OR REPLACE TEMP VIEW s31_case AS SELECT * FROM read_parquet({s31_q(S31_CASE)})')

summary = S31_DB.execute('''
SELECT income_recon_class, COUNT(*) AS n_cases,
       COUNT(*) FILTER (WHERE case_integrity_ok) AS n_integrity_passed,
       COUNT(*) FILTER (WHERE income_candidate_usable AND case_integrity_ok) AS n_income_candidates,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pct_cases
FROM s31_case GROUP BY 1 ORDER BY n_cases DESC
''').df()
display(summary)
summary.to_csv(S31_TABLES/'01_income_reconciliation.csv',index=False,encoding='utf-8-sig')

integrity = S31_DB.execute('''
SELECT COUNT(*) AS clean_cases,
       COUNT(*) FILTER (WHERE raw_records IS NULL) AS cases_missing_in_raw,
       COUNT(*) FILTER (WHERE NOT client_link_ok) AS client_link_failures,
       COUNT(*) FILTER (WHERE NOT status_match) AS status_mismatches,
       COUNT(*) FILTER (WHERE income_candidate_usable AND case_integrity_ok) AS usable_income_cases
FROM s31_case
''').df()
display(integrity)
integrity.to_csv(S31_TABLES/'02_case_integrity.csv',index=False,encoding='utf-8-sig')

,income_recon_class,n_cases,n_integrity_passed,n_income_candidates,pct_cases
0,CONSISTENT_POSITIVE,2320778,2320778,2320778,54.22
1,ZERO_OR_MISSING_PLUS_POSITIVE,1948498,1948498,1948498,45.52
2,NO_RAW_POSITIVE_INCOME,11101,11101,0,0.26


,clean_cases,cases_missing_in_raw,client_link_failures,status_mismatches,usable_income_cases
0,4280377,0,0,0,4269276


### Cell 4 — Bước 3.2: Tổng hợp nghĩa vụ và lịch sử thanh toán theo khách hàng

In [4]:
S31_DB.execute('''CREATE OR REPLACE TEMP VIEW s31_customer_counts AS
SELECT client_id,
       COUNT(*) AS n_all_cases,
       COUNT(*) FILTER (WHERE NOT case_integrity_ok) AS n_integrity_failed_cases,
       COUNT(*) FILTER (WHERE case_integrity_ok AND signed_status) AS n_signed_cases,
       COUNT(*) FILTER (WHERE case_integrity_ok AND credit_status='ACTIVE') AS a3_active_case_count,
       COUNT(*) FILTER (WHERE case_integrity_ok AND credit_status='ACTIVE'
                          AND annuity>0) AS n_active_cases_with_annuity,
       COUNT(*) FILTER (WHERE case_integrity_ok AND credit_status='ACTIVE'
                          AND (annuity IS NULL OR annuity<=0)) AS n_active_cases_missing_annuity,
       SUM(annuity) FILTER (WHERE case_integrity_ok AND credit_status='ACTIVE'
                             AND annuity>0) AS active_annuity_known,
       COUNT(*) FILTER (WHERE case_integrity_ok AND resolved_status_proxy AND due_flags_valid)
           AS n_payment_observable_resolved_proxy,
       COUNT(*) FILTER (WHERE case_integrity_ok AND resolved_status_proxy
                          AND due_flags_valid AND any_due=1) AS n_due_resolved_proxy,
       SUM(due_flag_count) FILTER (WHERE case_integrity_ok AND resolved_status_proxy
                                      AND due_flags_valid AND any_due=1) AS n_due_flags_affected_cases,
       COUNT(*) FILTER (WHERE case_integrity_ok AND signed_status AND due_flags_valid)
           AS n_payment_observable_signed_proxy,
       COUNT(*) FILTER (WHERE case_integrity_ok AND signed_status AND due_flags_valid
                          AND any_due=1) AS n_due_signed_proxy,
       COUNT(*) FILTER (WHERE case_integrity_ok AND credit_status='WRITTEN OFF') AS n_written_off_cases
FROM s31_case WHERE client_id IS NOT NULL GROUP BY client_id
''')

# Một case thu nhập có thể không đo được kỳ thu nhập; chọn ứng viên theo ngày nộp hồ sơ gần nhất.
S31_DB.execute('''CREATE OR REPLACE TEMP VIEW s31_income_candidates AS
SELECT client_id, case_id, product AS income_source_product,
       application_time AS income_source_application_time,
       reported_income AS latest_reported_income_candidate,
       income_recon_class AS latest_income_recon_class
FROM s31_case
WHERE case_integrity_ok AND income_candidate_usable
      AND signed_status AND application_time IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY client_id ORDER BY application_time DESC, case_id DESC
)=1
''')

S31_DB.execute('''CREATE OR REPLACE TEMP VIEW s31_income_span AS
SELECT client_id, COUNT(*) AS n_income_source_cases,
       MIN(reported_income) AS min_reported_income,
       MAX(reported_income) AS max_reported_income
FROM s31_case
WHERE case_integrity_ok AND income_candidate_usable AND signed_status
GROUP BY client_id
''')
print('Đã chuẩn bị 3 bảng tổng hợp customer-level (chưa tính score).')

Đã chuẩn bị 3 bảng tổng hợp customer-level (chưa tính score).


### Cell 5 — Kết xuất Feature Mart (6 ứng viên + cờ chất lượng)

In [5]:
S31_MART = S31_TABLES / '03_customer_feature_mart.csv'
S31_DB.execute(f'''
COPY (
WITH customers AS (
    SELECT TRIM(SKP_CLIENT) AS client_id FROM s31_clean_customer
), merged AS (
    SELECT b.client_id, a.* EXCLUDE(client_id),
           i.latest_reported_income_candidate AS income_candidate,
           i.income_source_application_time, i.income_source_product,
           i.latest_income_recon_class,
           h.n_income_source_cases, h.min_reported_income, h.max_reported_income
    FROM customers b
    LEFT JOIN s31_customer_counts a ON b.client_id=a.client_id
    LEFT JOIN s31_income_candidates i ON b.client_id=i.client_id
    LEFT JOIN s31_income_span h ON b.client_id=h.client_id
), features AS (
    SELECT *,
      CASE WHEN income_candidate>0 AND COALESCE(n_integrity_failed_cases,0)=0
                     AND COALESCE(n_active_cases_missing_annuity,0)=0
           THEN COALESCE(active_annuity_known,0)/income_candidate END
           AS A1_ODSR_IF_INCOME_MONTHLY,
      CASE WHEN income_candidate>0 AND COALESCE(n_integrity_failed_cases,0)=0
                     AND COALESCE(n_active_cases_missing_annuity,0)=0
           THEN income_candidate-COALESCE(active_annuity_known,0) END
           AS A2_RESIDUAL_IF_INCOME_MONTHLY,
      a3_active_case_count AS A3_ACTIVE_CASE_COUNT,
      CASE WHEN n_payment_observable_resolved_proxy>0
           THEN n_due_resolved_proxy::DOUBLE/n_payment_observable_resolved_proxy END
           AS P1_DUE_RATE_RESOLVED_PROXY,
      CASE WHEN n_due_resolved_proxy>0
           THEN n_due_flags_affected_cases::DOUBLE/n_due_resolved_proxy END
           AS P2_DUE_FLAG_BREADTH_AMONG_AFFECTED,
      CASE WHEN COALESCE(n_signed_cases,0)>0
           THEN CAST(n_written_off_cases>0 AS INTEGER) END AS P3_WRITTEN_OFF_HISTORY,
      CASE WHEN min_reported_income>0
           THEN max_reported_income/min_reported_income END AS income_spread_across_cases,
      (income_candidate>0 AND COALESCE(n_integrity_failed_cases,0)=0
            AND COALESCE(n_active_cases_missing_annuity,0)=0) AS capacity_data_complete_proxy,
      (COALESCE(n_payment_observable_resolved_proxy,0)>0) AS payment_observed_resolved_proxy,
      {str(S31_INCOME_IS_MONTHLY_CONFIRMED).upper()} AS income_period_monthly_confirmed
    FROM merged
)
SELECT client_id AS SKP_CLIENT, * FROM features ORDER BY client_id
) TO {s31_q(S31_MART)} (HEADER, DELIMITER ',')
''')
print('Feature mart:', S31_MART)
print('Size (MB):', round(S31_MART.stat().st_size/1024**2,1))

Feature mart: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\C1_STEP3_OUTPUT\run_20261018_091425_849a72\tables\03_customer_feature_mart.csv
Size (MB): 151.7


### Cell 6 — Báo cáo độ bao phủ và dữ liệu trước khi chọn 6 feature

In [6]:
S31_DB.execute(f'CREATE OR REPLACE TEMP VIEW s31_feature_mart AS SELECT * FROM read_csv({s31_q(S31_MART)}, header=true)')
coverage = S31_DB.execute('''
SELECT COUNT(*) AS n_customers,
       COUNT(*) FILTER (WHERE A1_ODSR_IF_INCOME_MONTHLY IS NOT NULL) AS n_a1_available,
       COUNT(*) FILTER (WHERE A2_RESIDUAL_IF_INCOME_MONTHLY IS NOT NULL) AS n_a2_available,
       COUNT(*) FILTER (WHERE A3_ACTIVE_CASE_COUNT IS NOT NULL) AS n_a3_available,
       COUNT(*) FILTER (WHERE P1_DUE_RATE_RESOLVED_PROXY IS NOT NULL) AS n_p1_available,
       COUNT(*) FILTER (WHERE P2_DUE_FLAG_BREADTH_AMONG_AFFECTED IS NOT NULL) AS n_p2_available,
       COUNT(*) FILTER (WHERE P3_WRITTEN_OFF_HISTORY IS NOT NULL) AS n_p3_available,
       COUNT(*) FILTER (WHERE A1_ODSR_IF_INCOME_MONTHLY IS NOT NULL
                          AND P1_DUE_RATE_RESOLVED_PROXY IS NOT NULL) AS n_capacity_and_payment,
       COUNT(*) FILTER (WHERE COALESCE(n_active_cases_missing_annuity,0)>0) AS n_incomplete_active_debt,
       COUNT(*) FILTER (WHERE income_candidate IS NULL) AS n_without_income_candidate,
       COUNT(*) FILTER (WHERE COALESCE(n_integrity_failed_cases,0)>0) AS n_integrity_issues
FROM s31_feature_mart
''').df()
display(coverage)
coverage.to_csv(S31_TABLES/'04_customer_feature_coverage.csv', index=False, encoding='utf-8-sig')

profile = S31_DB.execute('''
SELECT COALESCE(latest_income_recon_class,'NO_INCOME_CANDIDATE') AS income_quality,
       COUNT(*) AS n_customers,
       COUNT(*) FILTER (WHERE capacity_data_complete_proxy) AS capacity_complete_proxy,
       COUNT(*) FILTER (WHERE payment_observed_resolved_proxy) AS payment_proxy_available,
       COUNT(*) FILTER (WHERE income_spread_across_cases>1.5) AS income_spread_over_1_5
FROM s31_feature_mart GROUP BY 1 ORDER BY n_customers DESC
''').df()
display(profile)
profile.to_csv(S31_TABLES/'05_customer_income_quality.csv', index=False, encoding='utf-8-sig')

checks = S31_DB.execute('''
SELECT COUNT(*) AS n_customers,
       COUNT(DISTINCT client_id) AS n_unique_clients,
       COUNT(*) FILTER (WHERE client_id IS NULL) AS n_null_client,
       COUNT(*) FILTER (WHERE A1_ODSR_IF_INCOME_MONTHLY<0) AS n_negative_odsr,
       COUNT(*) FILTER (WHERE P1_DUE_RATE_RESOLVED_PROXY<0
                       OR P1_DUE_RATE_RESOLVED_PROXY>1) AS n_invalid_due_rates,
       COUNT(*) FILTER (WHERE P2_DUE_FLAG_BREADTH_AMONG_AFFECTED<1
                       OR P2_DUE_FLAG_BREADTH_AMONG_AFFECTED>4) AS n_invalid_due_breadth
FROM s31_feature_mart
''').df()
display(checks)
assert int(checks.n_customers.iloc[0])==int(checks.n_unique_clients.iloc[0])
assert checks[['n_null_client','n_negative_odsr','n_invalid_due_rates','n_invalid_due_breadth']].sum(axis=1).iloc[0]==0
checks.to_csv(S31_TABLES/'06_feature_validation.csv',index=False,encoding='utf-8-sig')

,n_customers,n_a1_available,n_a2_available,n_a3_available,n_p1_available,n_p2_available,n_p3_available,n_capacity_and_payment,n_incomplete_active_debt,n_without_income_candidate,n_integrity_issues
0,1000528,590696,590696,1000528,643574,36366,818023,471605,227234,182598,0


,income_quality,n_customers,capacity_complete_proxy,payment_proxy_available,income_spread_over_1_5
0,CONSISTENT_POSITIVE,443318,388316,367819,201977
1,ZERO_OR_MISSING_PLUS_POSITIVE,374612,202380,275664,135374
2,NO_INCOME_CANDIDATE,182598,0,91,0


,n_customers,n_unique_clients,n_null_client,n_negative_odsr,n_invalid_due_rates,n_invalid_due_breadth
0,1000528,1000528,0,0,0,0


### Cell 7 — Xuất sổ quy tắc và kết thúc

In [7]:
dictionary = pd.DataFrame([
    ['A1_ODSR_IF_INCOME_MONTHLY','Capacity','SUM(ANNUITY of Active)/latest positive reported income','Ứng viên; chỉ hợp lệ về tài chính khi thu nhập được xác nhận theo tháng và trạng thái cùng mốc quan sát.'],
    ['A2_RESIDUAL_IF_INCOME_MONTHLY','Capacity','Latest reported income - SUM(ANNUITY of Active)','Không trừ chi phí sinh hoạt, không phải disposable income.'],
    ['A3_ACTIVE_CASE_COUNT','Capacity','COUNT(DISTINCT active cases)','Nhiều hợp đồng không mặc nhiên là rủi ro.'],
    ['P1_DUE_RATE_RESOLVED_PROXY','Payment','Resolved proxy cases with ANY_DUE / resolved proxy cases with valid DUE','Mẫu số chỉ là trạng thái Finished/Paid off/Written off; chưa xác minh đủ 4 kỳ.'],
    ['P2_DUE_FLAG_BREADTH_AMONG_AFFECTED','Payment','SUM(number of binary DUE flags)/number of due-positive resolved proxy cases','Số cờ quan sát được; không phải xu hướng hay mức DPD. NULL nếu không có case DUE.'],
    ['P3_WRITTEN_OFF_HISTORY','Payment','1 if any case Written off; 0 otherwise among signed-status customers','Không có ngày sự kiện; 0 không chứng minh không có rủi ro.'],
    ['income_spread_across_cases','Quality','MAX(valid case income)/MIN(valid case income)','Biến động giữa hồ sơ, KHÔNG phải volatility thu nhập theo thời gian.'],
    ['capacity_data_complete_proxy','Quality','Income candidate and all Active have annuity','Không bảo đảm bao phủ nợ bên ngoài/BNPL/CC chưa có nghĩa vụ đo được.'],
    ['payment_observed_resolved_proxy','Quality','At least one resolved-status case with valid DUE flags','Không bảo đảm outcome maturity.'],
    ['income_period_monthly_confirmed','Quality','Manual documented setting','Mặc định FALSE; không tự giả định kỳ thu nhập.'],
],columns=['field','group','definition','limit'])
dictionary.to_csv(S31_TABLES/'07_feature_dictionary.csv',index=False,encoding='utf-8-sig')
display(dictionary)
print('\nHOÀN THÀNH BƯỚC 3.1 & 3.2')
print('Các file:',S31_TABLES)
print('Feature mart:',S31_MART)
print('Chưa tạo score hoặc trọng số; mọi công thức chưa xác minh được gắn nhãn proxy/IF_MONTHLY.')

,field,group,definition,limit
0,A1_ODSR_IF_INCOME_MONTHLY,Capacity,SUM(ANNUITY of Active)/latest positive reporte...,Ứng viên; chỉ hợp lệ về tài chính khi thu nhập...
1,A2_RESIDUAL_IF_INCOME_MONTHLY,Capacity,Latest reported income - SUM(ANNUITY of Active),"Không trừ chi phí sinh hoạt, không phải dispos..."
2,A3_ACTIVE_CASE_COUNT,Capacity,COUNT(DISTINCT active cases),Nhiều hợp đồng không mặc nhiên là rủi ro.
3,P1_DUE_RATE_RESOLVED_PROXY,Payment,Resolved proxy cases with ANY_DUE / resolved p...,Mẫu số chỉ là trạng thái Finished/Paid off/Wri...
4,P2_DUE_FLAG_BREADTH_AMONG_AFFECTED,Payment,SUM(number of binary DUE flags)/number of due-...,Số cờ quan sát được; không phải xu hướng hay m...
5,P3_WRITTEN_OFF_HISTORY,Payment,1 if any case Written off; 0 otherwise among s...,Không có ngày sự kiện; 0 không chứng minh khôn...
6,income_spread_across_cases,Quality,MAX(valid case income)/MIN(valid case income),"Biến động giữa hồ sơ, KHÔNG phải volatility th..."
7,capacity_data_complete_proxy,Quality,Income candidate and all Active have annuity,Không bảo đảm bao phủ nợ bên ngoài/BNPL/CC chư...
8,payment_observed_resolved_proxy,Quality,At least one resolved-status case with valid D...,Không bảo đảm outcome maturity.
9,income_period_monthly_confirmed,Quality,Manual documented setting,Mặc định FALSE; không tự giả định kỳ thu nhập.



HOÀN THÀNH BƯỚC 3.1 & 3.2
Các file: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\C1_STEP3_OUTPUT\run_20261018_091425_849a72\tables
Feature mart: C:\Users\nguyen\Desktop\BI_r2\BI10_ROUND02_DATASET\C1_STEP3_OUTPUT\run_20261018_091425_849a72\tables\03_customer_feature_mart.csv
Chưa tạo score hoặc trọng số; mọi công thức chưa xác minh được gắn nhãn proxy/IF_MONTHLY.
